# 07 LSTM Modelling

This notebook trains and evaluates an LSTM model using the current Dublin Bikes feature set and the current `band_30min` target definition.

> This notebook can be memory-intensive. If VS Code becomes unstable, use the optional station subset cell before sequence generation.

In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

PROJECT_ROOT = Path("..").resolve()
FEATURES_PATH = PROJECT_ROOT / "data" / "processed" / "dublin_bikes_features_with_trend.parquet"
if not FEATURES_PATH.exists():
    FEATURES_PATH = PROJECT_ROOT / "data" / "processed" / "dublin_bikes_features_with_trend.csv.gz"

if FEATURES_PATH.suffix == ".parquet":
    df = pd.read_parquet(FEATURES_PATH).copy()
else:
    df = pd.read_csv(FEATURES_PATH).copy()

df["last_reported"] = pd.to_datetime(df["last_reported"])
df = df.sort_values(["station_id", "last_reported"]).copy()

print("Loaded:", FEATURES_PATH)
print(df.shape)
df.head()


## Target creation

In [ ]:

def get_target(group):
    group = group.sort_values("last_reported").copy()
    sid = group["station_id"].iloc[0]

    times = group["last_reported"].to_numpy()
    values = group["utilisation"].to_numpy()

    target = np.full(len(group), np.nan)

    for i in range(len(times)):
        t_future = times[i] + np.timedelta64(30, "m")
        idx = np.searchsorted(times, t_future, side="left")
        if idx < len(times):
            target[i] = values[idx]

    group["utilisation_tplus_30"] = target
    group["station_id"] = sid
    return group

parts = []
for _, g in df.groupby("station_id", sort=False):
    parts.append(get_target(g))

df = pd.concat(parts, ignore_index=True)

def utilisation_band(u):
    if u < 0.05:
        return 0
    elif u < 0.30:
        return 1
    elif u < 0.70:
        return 2
    elif u < 0.95:
        return 3
    else:
        return 4

df["band_30min"] = df["utilisation_tplus_30"].apply(utilisation_band)
band_labels = [0, 1, 2, 3, 4]

print("Band distribution:")
print(df["band_30min"].value_counts(normalize=True).sort_index())


## Cleaning

In [ ]:

df = df.dropna(subset=["utilisation_tplus_30", "util_slope", "util_vs_mean"]).copy()
print(df.shape)


## LSTM features

In [ ]:

SEQ_FEATURES = [
    "utilisation",
    "roll_std_30min",
    "hour_slot",
    "is_weekday",
    "util_slope",
    "util_vs_mean",
]

TARGET_COL = "band_30min"
TIMESTEPS = 6

df_lstm = df.sort_values(["station_id", "last_reported"]).copy()
df_lstm = df_lstm.dropna(subset=SEQ_FEATURES + [TARGET_COL]).copy()

band_to_idx = {b: i for i, b in enumerate(sorted(df_lstm[TARGET_COL].unique()))}
idx_to_band = {i: b for b, i in band_to_idx.items()}
df_lstm["target_idx"] = df_lstm[TARGET_COL].map(band_to_idx)

print("Sequence features:", SEQ_FEATURES)
print("Classes:", band_to_idx)
print(df_lstm.shape)


## Optional: subset stations for memory stability

In [ ]:

# Uncomment this block if sequence creation is too heavy for your machine.
# max_stations = 40
# selected_stations = (
#     df_lstm["station_id"]
#     .drop_duplicates()
#     .sort_values()
#     .iloc[:max_stations]
# )
# df_lstm = df_lstm[df_lstm["station_id"].isin(selected_stations)].copy()
# print("Using subset shape:", df_lstm.shape)


## Sequence generation

In [ ]:

X_seq = []
y_seq = []

for station_id, group in df_lstm.groupby("station_id", sort=False):
    g = group.reset_index(drop=True)
    feat = g[SEQ_FEATURES].to_numpy(dtype=np.float32)
    tgt = g["target_idx"].to_numpy(dtype=np.int64)

    for i in range(len(g) - TIMESTEPS):
        X_seq.append(feat[i:i + TIMESTEPS])
        y_seq.append(tgt[i + TIMESTEPS])

X_seq = np.array(X_seq, dtype=np.float32)
y_seq = np.array(y_seq, dtype=np.int64)

print("Sequence array shape:", X_seq.shape)
print("Target array shape:", y_seq.shape)


## Time-ordered train/test split

In [ ]:

split_idx = int(len(X_seq) * 0.8)
X_train_seq, X_test_seq = X_seq[:split_idx], X_seq[split_idx:]
y_train_seq, y_test_seq = y_seq[:split_idx], y_seq[split_idx:]

print("Train sequences:", X_train_seq.shape, "Test sequences:", X_test_seq.shape)


## LSTM model

In [ ]:

num_classes = len(band_to_idx)

model = models.Sequential([
    layers.Input(shape=(TIMESTEPS, len(SEQ_FEATURES))),
    layers.LSTM(64, return_sequences=False),
    layers.Dense(64, activation="relu"),
    layers.Dense(num_classes, activation="softmax"),
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True,
)

history = model.fit(
    X_train_seq, y_train_seq,
    validation_split=0.1,
    epochs=5,
    batch_size=512,
    verbose=1,
    callbacks=[early_stop],
)


## Evaluation

In [ ]:

y_proba_lstm = model.predict(X_test_seq)
y_pred_lstm_idx = np.argmax(y_proba_lstm, axis=1)

print("LSTM Accuracy:", accuracy_score(y_test_seq, y_pred_lstm_idx))
print("\nLSTM Classification report:")
print(classification_report(
    y_test_seq,
    y_pred_lstm_idx,
    labels=list(range(num_classes)),
    target_names=[str(idx_to_band[i]) for i in range(num_classes)]
))


In [ ]:

cm = confusion_matrix(y_test_seq, y_pred_lstm_idx, labels=list(range(num_classes)))

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=[idx_to_band[i] for i in range(num_classes)],
    yticklabels=[idx_to_band[i] for i in range(num_classes)]
)
plt.title("LSTM — Confusion Matrix")
plt.ylabel("True Band")
plt.xlabel("Predicted Band")
plt.tight_layout()
plt.show()


In [ ]:

plt.figure(figsize=(8, 5))
plt.plot(history.history["loss"], label="train_loss")
plt.plot(history.history["val_loss"], label="val_loss")
plt.title("LSTM Training History")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.tight_layout()
plt.show()
